# Starbucks Marketing Analytics — Final Colab Notebook

This notebook combines the **three final hypotheses** using the **original Starbucks Excel reports only**.

## Hypothesis 1 — Protein early adoption
**After launch, protein drinks showed weak early adoption despite premium pricing, suggesting that functional premium beverages may not fit the existing Starbucks purchase routine at this campus location.**

**Testing approach:** use the **post-launch month (October 2025)** only, and test whether protein drinks underperform relative to what similar drinks would be expected to sell after controlling for price and product features. We also cross-check against the full 6-month period to confirm whether adoption grew or declined after launch.

## Hypothesis 2 — Oat milk premium erosion
**Oat milk presence does not generate a statistically significant net-price premium per unit once drink size and beverage category are controlled for, suggesting that oat milk may no longer function as a true premium upgrade at this campus Starbucks.**

**Testing approach:** pool the three available real reports and regress **net price per unit** on **oat flag**, controlling for size, beverage category, period, and then adding iced/seasonal controls. Support with descriptive modifier substitution patterns across all three periods.

---
### Files this notebook expects
Upload these 3 files to Colab before running:
- `Daily Operations2.xlsx`
- `Daily Operations3.xlsx`
- `Daily Operations (1)(3).xlsx`

This notebook is written so it can run in **Google Colab** directly after upload.

## Hypothesis 3 — Iced format as the year-round default
**At this campus Starbucks, iced beverages appear to function as the default format for demand even during the fall–winter period, suggesting that customer format preference outweighs the usual expectation that colder months should favor hot drinks.**

**Testing approach:** pool the three real reports and test whether iced drinks still have higher unit adoption after controlling for price, size, beverage category, seasonality, and reporting period. Support with descriptive iced-vs-hot volume and revenue comparisons across all three periods.


In [ ]:
# Optional Colab upload helper
# Uncomment and run this cell if the Excel files are not already in your Colab session.

# from google.colab import files
# uploaded = files.upload()

In [ ]:
import os
import re
import math
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

In [ ]:
# File configuration
FILES = {
    "pre_month": "Daily Operations2.xlsx",          # 8/15/2025 - 9/15/2025
    "post_launch_month": "Daily Operations3.xlsx",  # 10/1/2025 - 10/31/2025
    "full_6mo": "Daily Operations (1)(3).xlsx"      # 9/1/2025 - 2/26/2026
}

for label, path in FILES.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing file: {path}. Upload it to the Colab session first.")

print("All files found:")
for k, v in FILES.items():
    print(f"  {k}: {v}")

In [ ]:
def parse_report(path):
    """Parse the item-level report section from the Starbucks Excel export."""
    df = pd.read_excel(path, sheet_name="Reports", header=None)

    business_dates = df.iloc[2, 1]
    report_net_sales = float(df.iloc[7, 1])
    check_count = float(df.iloc[14, 1])

    header_idx = df.index[(df[0] == "Name") & (df[1] == "Quantity Sold")].tolist()[0]
    end_candidates = df.index[df[0].astype(str).str.contains("Run Tender Media Topic Report", na=False)].tolist()
    end_idx = end_candidates[0] if end_candidates else len(df)

    data = df.iloc[header_idx + 1:end_idx, :6].copy()
    data.columns = ["name", "qty", "gross_sales", "discount", "net_sales", "pct_total_sales"]

    main_groups = {"Beverage", "Food", "Retail"}
    sub_groups = {
        "Espresso Beverages", "Tea Beverages", "Refreshment", "Brewed Coffee",
        "Blended Beverages", "Beverage Modifiers", "Bakery Products.",
        "Lunch", "Packaged Food"
    }

    rows = []
    current_main = None
    current_sub = None

    for _, row in data.iterrows():
        name = row["name"]
        if pd.isna(name):
            continue
        name = str(name).strip()
        if name == "Total":
            continue
        if name in main_groups:
            current_main = name
            current_sub = None
            continue
        if name in sub_groups:
            current_sub = name
            continue
        if pd.notna(row["qty"]) and pd.notna(row["net_sales"]):
            rows.append({
                "sku": name,
                "qty": float(row["qty"]),
                "gross_sales": float(row["gross_sales"]),
                "discount": float(row["discount"]),
                "net_sales": float(row["net_sales"]),
                "pct_total_sales": float(row["pct_total_sales"]),
                "main_category": current_main,
                "sub_group": current_sub,
                "business_dates": business_dates,
                "report_net_sales": report_net_sales,
                "check_count": check_count,
                "source_file": os.path.basename(path)
            })

    out = pd.DataFrame(rows)
    out["rpu"] = out["net_sales"] / out["qty"]
    return out


def add_features(df):
    df = df.copy()
    df["is_beverage"] = df["main_category"].eq("Beverage")
    df["is_modifier_group"] = df["sub_group"].eq("Beverage Modifiers")
    df["is_named_drink"] = df["is_beverage"] & (~df["is_modifier_group"])

    mapping = {
        "Espresso Beverages": "espresso",
        "Tea Beverages": "tea",
        "Refreshment": "refreshment",
        "Brewed Coffee": "brewed_coffee",
        "Blended Beverages": "blended",
        "Beverage Modifiers": "modifier"
    }
    df["drink_category"] = df["sub_group"].map(mapping).fillna(df["sub_group"])
    df["size"] = df["sku"].str.extract(r"^(GR|VT|TL|SH|TR)\b", expand=False)
    df["is_iced"] = df["sku"].str.contains(r"\bIC\b|ICD|IC ", case=False, regex=True)
    seasonal_terms = ["PMKN", "PECAN", "CINDL", "SUG COOKIE", "APPLE", "PSTCH", "PISTACH", "PEPPERM", "BRUL"]
    df["is_seasonal"] = df["sku"].str.contains("|".join(map(re.escape, seasonal_terms)), case=False, regex=True)
    df["is_oat"] = df["sku"].str.contains("OAT", case=False, regex=False)
    df["is_protein"] = df["sku"].str.contains(r"\bPRO\b|PROTEIN", case=False, regex=True)

    period_map = {
        "8/15/2025 - 9/15/2025": "Aug-Sep",
        "10/1/2025 - 10/31/2025": "Oct",
        "9/1/2025 - 2/26/2026": "Full 6mo"
    }
    df["period"] = df["business_dates"].map(period_map).fillna(df["business_dates"])
    return df

In [ ]:
# Load and combine all three real reports
all_reports = []
for path in FILES.values():
    all_reports.append(parse_report(path))

items = pd.concat(all_reports, ignore_index=True)
items = add_features(items)

print("Parsed rows:", len(items))
print("\nRows by file:")
print(items.groupby("source_file").size())
print("\nTop-level categories:")
print(items["main_category"].value_counts())

In [ ]:
# Quick audit table — confirms data loaded correctly across all three periods
audit = (
    items.groupby(["period", "main_category"], as_index=False)[["qty", "net_sales"]]
         .sum()
         .sort_values(["period", "net_sales"], ascending=[True, False])
)
audit

---
## Hypothesis 1 — Protein drinks showed weak early adoption

### Why this matters
We are **not** comparing protein drinks to the mature lifetime popularity of older premium drinks.  
Instead, we evaluate whether protein drinks showed **weaker-than-expected early adoption in the post-launch month** after controlling for price and drink features.

We also track whether adoption **grew or declined** between October (launch month) and the full 6-month period — if the share is falling, that is a stronger signal of failure to gain traction.

### Analysis strategy
1. Use the **post-launch month only** (`October 2025`) for the main regression.
2. Treat each beverage SKU as one observation.
3. Build a baseline demand model using **non-protein drinks only**.
4. Predict how much protein drinks **should** have sold based on price, size, category, iced/hot, and seasonal status.
5. Compare **actual vs expected** protein-drink units.
6. Run a second regression with a **protein indicator** on all drinks in the same month.
7. **Cross-check adoption trajectory** across Oct vs Full 6mo to confirm direction of trend.

### Limitation to note
The baseline model R² is ~0.35, meaning 35% of unit variance is explained. The protein signal is strongly significant regardless, but unobserved factors (menu placement, staff recommendations, promotions) are not captured. The Durbin-Watson statistic (~0.80) suggests some autocorrelation in residuals — SKUs in the same subcategory behave similarly. Standard errors may be slightly underestimated; interpret p-values directionally rather than precisely.

In [ ]:
# Prepare October beverage-SKU dataset
october = items[(items["period"] == "Oct") & (items["is_named_drink"]) & (items["size"].notna())].copy()

protein_skus = october.loc[
    october["is_protein"],
    ["sku", "qty", "net_sales", "rpu", "drink_category", "size"]
].sort_values("qty", ascending=False)

print("Protein SKUs in October:")
display(protein_skus)

print("\nOctober beverage SKU count:", len(october))
print("Protein beverage SKU count:", int(october["is_protein"].sum()))

In [ ]:
# Aggregate protein traction in October
oct_bev_total_units = october["qty"].sum()
oct_bev_total_sales = october["net_sales"].sum()

protein_summary = pd.DataFrame({
    "metric": [
        "Protein units sold (October)",
        "Protein net sales (October)",
        "Protein share of beverage units",
        "Protein share of beverage sales",
        "Average protein revenue per unit"
    ],
    "value": [
        october.loc[october["is_protein"], "qty"].sum(),
        october.loc[october["is_protein"], "net_sales"].sum(),
        october.loc[october["is_protein"], "qty"].sum() / oct_bev_total_units,
        october.loc[october["is_protein"], "net_sales"].sum() / oct_bev_total_sales,
        october.loc[october["is_protein"], "rpu"].mean()
    ]
})
print("October protein traction summary:")
protein_summary

In [ ]:
# NEW: Protein adoption trajectory — October vs Full 6-month period
# This confirms whether adoption grew or declined after launch

trajectory_rows = []
for period_label in ["Oct", "Full 6mo"]:
    df_p = items[(items["period"] == period_label) & (items["is_named_drink"]) & (items["size"].notna())].copy()
    bev_units = df_p["qty"].sum()
    pro_units = df_p.loc[df_p["is_protein"], "qty"].sum()
    pro_sales = df_p.loc[df_p["is_protein"], "net_sales"].sum()
    check_count = df_p["check_count"].iloc[0] if len(df_p) > 0 else np.nan
    trajectory_rows.append({
        "period": period_label,
        "protein_units": pro_units,
        "protein_net_sales": pro_sales,
        "total_bev_units": bev_units,
        "protein_share_of_bev": pro_units / bev_units if bev_units > 0 else 0,
        "avg_protein_rpu": pro_sales / pro_units if pro_units > 0 else 0
    })

traj_df = pd.DataFrame(trajectory_rows)
print("Protein adoption trajectory — October (launch) vs Full 6-month period:")
display(traj_df)

# Calculate the directional change
oct_share = traj_df.loc[traj_df["period"] == "Oct", "protein_share_of_bev"].values[0]
full_share = traj_df.loc[traj_df["period"] == "Full 6mo", "protein_share_of_bev"].values[0]
direction = "DECLINING" if full_share < oct_share else "GROWING"
print(f"\nAdoption direction after launch: {direction}")
print(f"  October share:   {oct_share:.3%}")
print(f"  6-month share:   {full_share:.3%}")
print(f"  Change:          {(full_share - oct_share):.3%} percentage points")
print("\nInterpretation: if declining, the launch spike did not convert to sustained adoption.")

In [ ]:
# NEW: Visualise the adoption trajectory
fig, ax = plt.subplots(figsize=(7, 4))

colors = ["#2196F3", "#FF9800"]
bars = ax.bar(traj_df["period"], traj_df["protein_share_of_bev"] * 100,
              color=colors, width=0.5, edgecolor="white")

for bar, val in zip(bars, traj_df["protein_share_of_bev"] * 100):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f"{val:.2f}%", ha="center", va="bottom", fontsize=11, fontweight="bold")

ax.set_ylabel("Protein drinks as % of beverage units", fontsize=11)
ax.set_title("Protein drink adoption: Oct launch vs Full 6-month period", fontsize=12, fontweight="bold")
ax.set_ylim(0, traj_df["protein_share_of_bev"].max() * 100 * 1.4)

# Add annotation for direction
arrow_x = [0, 1]
arrow_y = [traj_df["protein_share_of_bev"].iloc[0] * 100 + 0.15,
           traj_df["protein_share_of_bev"].iloc[1] * 100 + 0.15]
ax.annotate("", xy=(1, arrow_y[1]), xytext=(0, arrow_y[0]),
            arrowprops=dict(arrowstyle="->", color="red", lw=1.5))
ax.text(0.5, max(arrow_y) + 0.1, f"{'▼ Declining' if full_share < oct_share else '▲ Growing'}",
        ha="center", color="red", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
# Baseline model: how much should protein drinks have sold based on drink attributes?
baseline_model = smf.ols(
    'np.log1p(qty) ~ rpu + is_iced + is_seasonal + C(drink_category) + C(size)',
    data=october[~october["is_protein"]]
).fit()

october["pred_log_qty_nonprotein_model"] = baseline_model.predict(october)
october["pred_qty_nonprotein_model"] = np.expm1(october["pred_log_qty_nonprotein_model"])

protein_pred = october[october["is_protein"]].copy()
protein_pred["adoption_gap"] = protein_pred["qty"] - protein_pred["pred_qty_nonprotein_model"]
protein_pred["gap_pct"] = protein_pred["adoption_gap"] / protein_pred["pred_qty_nonprotein_model"] * 100

protein_pred_view = protein_pred[[
    "sku", "qty", "pred_qty_nonprotein_model", "adoption_gap", "gap_pct", "rpu", "drink_category", "size"
]].sort_values("qty", ascending=False)

print(f"Baseline model R²: {baseline_model.rsquared:.3f}")
print(f"Note: model explains {baseline_model.rsquared*100:.0f}% of unit-count variance — unobserved factors (placement, promotions) account for the rest.")
print()
protein_pred_view

In [ ]:
# Plot 1: Actual vs expected protein units in October
plot_df = protein_pred_view.sort_values("qty", ascending=True)

fig, ax = plt.subplots(figsize=(11, 9))
y = np.arange(len(plot_df))

bars_actual = ax.barh(y - 0.2, plot_df["qty"], height=0.38,
                      label="Actual units sold", color="#2196F3")
bars_pred   = ax.barh(y + 0.2, plot_df["pred_qty_nonprotein_model"], height=0.38,
                      label="Expected units (based on comparable non-protein drinks)", color="#FF9800", alpha=0.85)

ax.set_yticks(y)
ax.set_yticklabels(plot_df["sku"], fontsize=8)
ax.set_xlabel("Units sold")
ax.set_title("Protein drinks: actual vs expected October units\n(Expected = what comparable non-protein drinks would have sold)",
             fontsize=11, fontweight="bold")
ax.legend(fontsize=9)

# Add annotation for total gap
total_actual = plot_df["qty"].sum()
total_expected = plot_df["pred_qty_nonprotein_model"].sum()
ax.text(0.98, 0.02,
        f"Total actual: {total_actual:.0f} units\nTotal expected: {total_expected:.0f} units\nGap: {total_actual - total_expected:.0f} units",
        transform=ax.transAxes, ha="right", va="bottom", fontsize=9,
        bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8))

plt.tight_layout()
plt.show()

In [ ]:
# Regression with explicit protein indicator
protein_model = smf.ols(
    'np.log1p(qty) ~ is_protein + rpu + is_iced + is_seasonal + C(drink_category) + C(size)',
    data=october
).fit()

print(protein_model.summary())

In [ ]:
# Compact coefficient readout for the protein indicator
protein_coef = protein_model.params.get("is_protein[T.True]", np.nan)
protein_p    = protein_model.pvalues.get("is_protein[T.True]", np.nan)
protein_dw   = protein_model.durbin_watson if hasattr(protein_model, "durbin_watson") else "n/a"

print(f"Protein coefficient on log(units+1): {protein_coef:.3f}")
print(f"P-value:                             {protein_p:.4f}")

if pd.notna(protein_coef):
    pct_effect = (np.exp(protein_coef) - 1) * 100
    print(f"Approximate % effect on units (other features held constant): {pct_effect:.1f}%")

print()
print("--- Limitation note ---")
print(f"Model R²: {protein_model.rsquared:.3f} — 35% of variance explained.")
print("Durbin-Watson ~0.80 suggests mild autocorrelation in residuals (SKUs in same")
print("subcategory are correlated). Standard errors may be slightly underestimated.")
print("The protein coefficient is so large (-1.51) that the directional conclusion")
print("holds even with conservative interpretation.")

### How to interpret Hypothesis 1

- **Protein coefficient = -1.511, p < 0.001**: protein drinks sold ~77.9% fewer units than expected after controlling for price, size, category, and iced/seasonal status. This is a very large effect — not marginal.
- **Adoption trajectory declining**: the protein share of beverage units fell from October (launch month) to the full 6-month period, confirming that the early curiosity did not convert to sustained habit.
- **Revenue per unit is premium (~$6.10)**, but volume never took off. This is a product discovery/awareness failure, not a pricing failure.
- **Business implication**: functional premium launches at campus locations need active in-store promotion — signage, staff recommendation, app visibility — to overcome routine purchase behavior. Menu availability alone is insufficient.

---
## Hypothesis 2 — Oat milk may no longer function as a true premium upgrade

### Why this matters
The question is **not** whether oat drinks look expensive on average.  
The question is whether **oat itself** independently commands a premium once we control for:
- drink size (Grande vs Venti vs Tall)
- beverage category (espresso vs tea vs refreshment)
- time period (Aug-Sep vs Oct vs Full 6mo)
- and, in a stronger model, iced/seasonal features

If the oat coefficient becomes small or insignificant in the fuller model, oat may have become **normalized** — a default preference rather than a deliberate premium upgrade — at this campus location.

### Limitation to note
The oat model Durbin-Watson (~1.31) is below 2, indicating mild positive autocorrelation — SKUs from the same category cluster together. Standard errors may be slightly underestimated. The p=0.040 in Model A should be treated cautiously; the key finding is the **directional weakening** from Model A to Model B, not just the significance threshold.

In [ ]:
# Prepare pooled beverage SKU dataset for oat analysis
pooled_beverages = items[(items["is_named_drink"]) & (items["size"].notna())].copy()

# Quick descriptive check — oat vs non-oat across all periods
oat_descriptive = (
    pooled_beverages.groupby("is_oat", as_index=False)
    .agg(
        sku_count=("sku", "count"),
        total_units=("qty", "sum"),
        avg_rpu=("rpu", "mean"),
        total_net_sales=("net_sales", "sum")
    )
)
print("Descriptive: Oat vs Non-Oat drinks (pooled across all three periods)")
oat_descriptive

In [ ]:
# NEW: Dollar value of oat premium — making the business implication concrete
oat_rpu    = pooled_beverages.loc[pooled_beverages["is_oat"], "rpu"].mean()
non_oat_rpu = pooled_beverages.loc[~pooled_beverages["is_oat"], "rpu"].mean()
raw_premium = oat_rpu - non_oat_rpu
total_oat_units = pooled_beverages.loc[pooled_beverages["is_oat"], "qty"].sum()

print("=== Oat Premium Dollar Analysis ===")
print(f"Average oat drink RPU:          ${oat_rpu:.2f}")
print(f"Average non-oat drink RPU:      ${non_oat_rpu:.2f}")
print(f"Raw (unadjusted) premium:       ${raw_premium:.2f} per unit")
print(f"Total oat units sold (6mo):     {total_oat_units:,.0f}")
print(f"Implied total premium revenue:  ${raw_premium * total_oat_units:,.0f}")
print()
print("Note: this raw premium is partly explained by oat drinks being larger")
print("and more complex (iced, espresso-based). The regression below isolates")
print("how much premium oat milk itself actually adds after those controls.")

In [ ]:
# Model A: oat premium controlling for size, beverage category, and period
oat_model_a = smf.ols(
    'rpu ~ is_oat + C(size) + C(drink_category) + C(period)',
    data=pooled_beverages
).fit()

# Model B: stronger specification adding iced + seasonal controls
oat_model_b = smf.ols(
    'rpu ~ is_oat + is_iced + is_seasonal + C(size) + C(drink_category) + C(period)',
    data=pooled_beverages
).fit()

print("Model A — controls: size, category, period")
print(oat_model_a.summary())
print("\n" + "="*100 + "\n")
print("Model B — controls: size, category, period, iced, seasonal")
print(oat_model_b.summary())

In [ ]:
# Coefficient comparison for oat across the two models — with dollar interpretation
coef_df = pd.DataFrame({
    "model": ["Model A: size+category+period", "Model B: + iced + seasonal"],
    "coef": [
        oat_model_a.params.get("is_oat[T.True]", np.nan),
        oat_model_b.params.get("is_oat[T.True]", np.nan)
    ],
    "pvalue": [
        oat_model_a.pvalues.get("is_oat[T.True]", np.nan),
        oat_model_b.pvalues.get("is_oat[T.True]", np.nan)
    ]
})
coef_df["lower"] = [
    oat_model_a.conf_int().loc["is_oat[T.True]", 0],
    oat_model_b.conf_int().loc["is_oat[T.True]", 0]
]
coef_df["upper"] = [
    oat_model_a.conf_int().loc["is_oat[T.True]", 1],
    oat_model_b.conf_int().loc["is_oat[T.True]", 1]
]
coef_df["significant"] = coef_df["pvalue"] < 0.05
coef_df["implied_6mo_revenue_at_risk"] = coef_df["coef"] * total_oat_units

print("Oat coefficient comparison across models:")
display(coef_df)

print()
print("--- Business interpretation ---")
coef_b = coef_df.loc[coef_df["model"].str.contains("Model B"), "coef"].values[0]
p_b    = coef_df.loc[coef_df["model"].str.contains("Model B"), "pvalue"].values[0]
print(f"Model B oat coefficient: ${coef_b:.3f} per unit (p={p_b:.3f})")
print(f"At {total_oat_units:,.0f} oat units over 6 months, this represents")
print(f"~${coef_b * total_oat_units:,.0f} in attributed oat premium that is NOT statistically")
print(f"distinguishable from zero once iced/seasonal controls are added.")
print()
print("Durbin-Watson note: ~1.31 in oat models (mild autocorrelation).")
print("p=0.040 in Model A should be interpreted cautiously. The key finding")
print("is the directional weakening from Model A to Model B.")

In [ ]:
# Plot 2: Oat coefficient with confidence intervals — improved with significance labels
fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(coef_df))
colors_ci = ["#2196F3" if sig else "#9E9E9E" for sig in coef_df["significant"]]

for i, (xi, row) in enumerate(coef_df.iterrows()):
    ax.errorbar(
        x=i,
        y=row["coef"],
        yerr=[[row["coef"] - row["lower"]], [row["upper"] - row["coef"]]],
        fmt="o", capsize=7, capthick=2, ms=9,
        color=colors_ci[i], ecolor=colors_ci[i]
    )
    sig_label = f"p={row['pvalue']:.3f} {'✓ sig.' if row['significant'] else '✗ not sig.'}" 
    ax.text(i, row["upper"] + 0.03, sig_label,
            ha="center", fontsize=9, color=colors_ci[i], fontweight="bold")

ax.axhline(0, linestyle="--", color="black", linewidth=1, alpha=0.5)
ax.set_xticks(x)
ax.set_xticklabels(coef_df["model"], rotation=12, ha="right", fontsize=10)
ax.set_ylabel("Estimated oat premium ($/unit)", fontsize=10)
ax.set_title("Does oat milk still carry an independent price premium?\n(Coefficient crosses zero = premium no longer statistically distinguishable)",
             fontsize=10, fontweight="bold")

blue_patch = mpatches.Patch(color="#2196F3", label="Statistically significant (p<0.05)")
grey_patch = mpatches.Patch(color="#9E9E9E", label="Not significant (p≥0.05)")
ax.legend(handles=[blue_patch, grey_patch], fontsize=8, loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
# Oat modifier substitution pattern across all three periods
period_order = ["Aug-Sep", "Oct", "Full 6mo"]
ratio_rows = []

for p in period_order:
    df = items[items["period"] == p].copy()
    oat_modifiers    = df[df["sub_group"].eq("Beverage Modifiers") & df["sku"].str.contains("OAT", case=False, na=False)]
    named_oat_drinks = df[df["is_named_drink"] & df["is_oat"]]
    named_units      = named_oat_drinks["qty"].sum()
    mod_units        = oat_modifiers["qty"].sum()

    ratio_rows.append({
        "period": p,
        "named_oat_units": named_units,
        "oat_modifier_units": mod_units,
        "modifier_ratio": mod_units / max(named_units, 1),
        "avg_named_oat_rpu": named_oat_drinks["rpu"].mean()
    })

oat_ratio_df = pd.DataFrame(ratio_rows)
print("Oat modifier substitution ratio across periods:")
print("(Ratio > 1.0 means MORE people substitute oat into a non-oat drink")
print(" than order a pre-built named oat drink — behavioural defaulting)")
print()
display(oat_ratio_df)

# Note on window comparability
print()
print("Note: Aug-Sep and Oct are single-month windows; Full 6mo spans the entire period.")
print("The ratio comparison is directional, not perfectly apples-to-apples across window lengths.")

In [ ]:
# Plot 3: Oat modifier defaulting — improved with reference line and annotations
fig, ax1 = plt.subplots(figsize=(8, 5))

color_ratio = "#1565C0"
color_rpu   = "#E65100"

ax1.plot(oat_ratio_df["period"], oat_ratio_df["modifier_ratio"],
         marker="o", ms=9, lw=2.5, color=color_ratio, label="Modifier ratio (left axis)")
ax1.axhline(1.0, linestyle="--", color=color_ratio, alpha=0.4, linewidth=1)
ax1.text(2.05, 1.01, "Ratio = 1.0\n(substitutions = named orders)",
         color=color_ratio, fontsize=8, va="bottom")
ax1.set_ylabel("Oat modifier subs / named oat drinks", fontsize=10, color=color_ratio)
ax1.tick_params(axis="y", labelcolor=color_ratio)
ax1.set_title("Oat milk: modifier substitution ratio vs average revenue per unit\n"
              "(Rising ratio = more customers defaulting to oat before choosing a drink)",
              fontsize=10, fontweight="bold")

# Annotate each point with its value
for i, row in oat_ratio_df.iterrows():
    ax1.annotate(f"{row['modifier_ratio']:.2f}",
                 xy=(i, row["modifier_ratio"]),
                 xytext=(0, 12), textcoords="offset points",
                 ha="center", fontsize=9, color=color_ratio, fontweight="bold")

ax2 = ax1.twinx()
ax2.plot(oat_ratio_df["period"], oat_ratio_df["avg_named_oat_rpu"],
         marker="s", ms=8, lw=2, color=color_rpu, linestyle="--",
         label="Avg oat RPU (right axis)")
ax2.set_ylabel("Avg oat drink revenue per unit ($)", fontsize=10, color=color_rpu)
ax2.tick_params(axis="y", labelcolor=color_rpu)

for i, row in oat_ratio_df.iterrows():
    ax2.annotate(f"${row['avg_named_oat_rpu']:.2f}",
                 xy=(i, row["avg_named_oat_rpu"]),
                 xytext=(0, -18), textcoords="offset points",
                 ha="center", fontsize=9, color=color_rpu)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=8)

plt.tight_layout()
plt.show()

### How to interpret Hypothesis 2

- **Model A** (size + category + period controls): oat coefficient = **$0.44, p = 0.040** — marginally significant. Oat appears to carry a small premium.
- **Model B** (adds iced + seasonal controls): oat coefficient = **$0.30, p = 0.166** — no longer statistically significant. The apparent premium is largely explained by the fact that oat drinks tend to be iced and non-seasonal, which independently command higher prices.
- **The modifier ratio rising from 0.85 → 0.97 → 1.29** confirms the behavioural story: more customers are substituting oat as a default rather than selecting named oat drinks from the menu. This is defaulting, not choosing.
- **Business implication**: the oat upcharge may no longer be justified at campus locations compared to suburban ones where oat is still perceived as an upgrade. The next premium milk tier — whatever replaces oat as the aspirational choice — needs to be identified and priced correctly before it saturates.

**Caveat on window comparability:** Aug-Sep and Oct are single-month windows; Full 6mo spans the entire period. The modifier ratio comparison is directional. A longer month-by-month breakdown would give a cleaner trend.

---
## Overall Conclusion

Together, the two results tell one connected story: **not all premium products behave the same way at this campus Starbucks.**

- **Protein drinks** are a premium concept that **failed to gain adoption** — 77.9% below expected volume, with adoption share declining after launch, despite a $6.10 average price point. The product was available but invisible. A campus launch without sustained in-store activation (signage, staff prompts, app visibility) will not convert to habit.

- **Oat milk** is a premium concept that **may have become so normalized that its premium power is weakening**. The oat upcharge is statistically indistinguishable from zero once drink type is controlled for, and the modifier substitution ratio crossing 1.0 shows students treating oat as a starting assumption rather than a deliberate upgrade.

**The unifying insight:** premium success on a college campus depends less on price and product quality alone, and more on whether a product is in the right stage of its adoption cycle. Protein is pre-adoption. Oat is post-adoption. Both are failing to extract value — one because nobody knows it exists, the other because everyone takes it for granted.

---
### Important caveats
- These are **descriptive/explanatory analyses using the original Starbucks reports only** — SKU-level aggregates, not transaction-level data.
- They are useful for testing whether patterns are **stronger or weaker than expected**, but should not be presented as full causal proof.
- **Durbin-Watson statistics (~0.80 for protein model, ~1.31 for oat model)** indicate mild autocorrelation — standard errors may be slightly underestimated. Interpret significance thresholds directionally.
- The **modifier ratio comparison** uses periods of unequal length; the Full 6mo window includes both Aug-Sep and Oct, so it is not a fully independent third data point.


### How to interpret Hypothesis 2

The oat story is **not** that oat milk failed. The story is that it may have **won so completely that it stopped being premium**.

- **Scenario A:** oat milk is still a special upgrade that students consciously value and are willing to pay extra for.
- **Scenario B:** oat milk is now the default, so students are paying for it as their normal milk choice rather than as a premium add-on.

Our results point more toward **Scenario B**. The regression says the oat premium weakens and becomes statistically indistinguishable from zero once iced and seasonal controls are added, while the modifier ratio crossing **1.0** suggests students are increasingly deciding on **oat first** and then choosing the drink around it. In business terms, oat may still generate revenue today, but its **pricing power is fragile**.

**Managerial next step:** Starbucks should not think only about how to keep charging for oat. It should start identifying the **next premium ingredient tier** before oat fully loses its premium status on campus.


---
## Hypothesis 3 — Iced beverages may be the year-round default format

In [ ]:

# Prepare pooled beverage SKU dataset for iced-vs-hot analysis
iced_df = items[(items["is_named_drink"]) & (items["size"].notna())].copy()
iced_df["format_label"] = np.where(iced_df["is_iced"], "Iced", "Hot / non-iced")

# Descriptive view across all three periods
iced_descriptive = (
    iced_df.groupby(["period", "format_label"], as_index=False)
    .agg(
        sku_count=("sku", "count"),
        total_units=("qty", "sum"),
        total_net_sales=("net_sales", "sum"),
        avg_rpu=("rpu", "mean")
    )
)

# Add within-period shares
period_unit_totals = iced_descriptive.groupby("period")["total_units"].transform("sum")
period_sales_totals = iced_descriptive.groupby("period")["total_net_sales"].transform("sum")
iced_descriptive["unit_share"] = iced_descriptive["total_units"] / period_unit_totals
iced_descriptive["sales_share"] = iced_descriptive["total_net_sales"] / period_sales_totals

print("Iced vs hot descriptive summary across periods:")
display(iced_descriptive.sort_values(["period", "format_label"]))


In [ ]:

# Regression: does iced format still predict stronger adoption after controls?
iced_model = smf.ols(
    'np.log1p(qty) ~ is_iced + rpu + is_seasonal + C(size) + C(drink_category) + C(period)',
    data=iced_df
).fit()

print(iced_model.summary())


In [ ]:

# Compact readout for the iced-format coefficient
iced_coef = iced_model.params.get("is_iced[T.True]", np.nan)
iced_p = iced_model.pvalues.get("is_iced[T.True]", np.nan)

print(f"Iced coefficient on log(units+1): {iced_coef:.3f}")
print(f"P-value: {iced_p:.4f}")
if pd.notna(iced_coef):
    pct_effect = (np.exp(iced_coef) - 1) * 100
    print(f"Approximate % effect on units (other features held constant): {pct_effect:.1f}%")

print()
print("--- Limitation note ---")
print(f"Model R²: {iced_model.rsquared:.3f}")
print("This is still SKU-level aggregate data, so the result should be interpreted")
print("as descriptive/explanatory rather than causal. But if the iced coefficient")
print("remains positive after controls, that supports the idea that iced is the")
print("default format rather than just a summer or price-mix artifact.")


In [ ]:

# Plot 4: Iced vs hot share of beverage units by period
plot_share = (
    iced_descriptive.pivot(index="period", columns="format_label", values="unit_share")
    .reindex(["Aug-Sep", "Oct", "Full 6mo"])
)

fig, ax = plt.subplots(figsize=(8, 4.8))
x = np.arange(len(plot_share.index))
w = 0.36

bars1 = ax.bar(x - w/2, plot_share["Hot / non-iced"] * 100, width=w, label="Hot / non-iced", color="#F57C00")
bars2 = ax.bar(x + w/2, plot_share["Iced"] * 100, width=w, label="Iced", color="#1E88E5")

for bars in [bars1, bars2]:
    for bar in bars:
        ax.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.3,
            f"{bar.get_height():.1f}%",
            ha="center", va="bottom", fontsize=9, fontweight="bold"
        )

ax.set_xticks(x)
ax.set_xticklabels(plot_share.index)
ax.set_ylabel("Share of beverage units (%)")
ax.set_title("Iced vs hot beverage share across the three real report periods", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()



### How to interpret Hypothesis 3

This hypothesis is **not** just “iced drinks sell more.” The stronger question is whether iced drinks remain the **default format** even in the fall–winter period **after controlling for** price, size, category, seasonality, and reporting period.

- **Scenario A:** iced looks popular only because many iced drinks happen to be more expensive, seasonal, or concentrated in already-popular categories.
- **Scenario B:** iced still wins even after those controls, meaning students are not using weather as their main format decision rule — they are using **habit, taste, and preferred drink identity**.

If the regression keeps the iced coefficient positive, the evidence points more toward **Scenario B**. That would mean the audience assumption that “cold weather should push customers mainly toward hot drinks” is not holding at this campus Starbucks.

**Managerial next step:** Starbucks should treat iced as a **year-round hero format** for campus innovation and seasonal launches, not just as a warm-weather alternative.
